In [ ]:
#for google colab
from google.colab import drive
drive.mount('/content/drive')
import json
import pandas as pd

with open('/content/drive/MyDrive/uberproject1/orders.json') as f:
    data = json.load(f)

df_orders = pd.DataFrame(data)
print(df_orders.shape)
print(df_orders.head())

(25000, 6)
                               order_id          restaurant_name  order_date  \
0  8174c2af-07a4-4837-9068-1169d963e36e           TBC Sky Lounge  2025-05-06   
1  0f9ddb57-4632-4a9f-9afe-1d2e41b94e32  KC Das - Sweet Paradise  2026-01-08   
2  099deda6-8c53-41cb-abf8-00126e6b2643        Hotel Kadamba Veg  2026-01-22   
3  94a6ba27-d6ef-4d10-a026-ac0c0afa8505            Cafe Srinidhi  2025-11-10   
4  216f3ec7-80a9-4d95-872f-715c37163b6d                   Pebble  2025-12-22   

   order_value discount_used payment_method  
0       201.87            No           Card  
1      1392.27            No           Cash  
2      1358.35           Yes            UPI  
3       782.37            No            UPI  
4      1551.09           Yes            UPI

In [ ]:
print(df_orders.dtypes)
print(df_orders.isnull().sum())
print(df_orders.duplicated().sum())
print(df_orders['payment_method'].value_counts())
print(df_orders['discount_used'].value_counts())

order_id            object
restaurant_name     object
order_date          object
order_value        float64
discount_used       object
payment_method      object
dtype: object
order_id           0
restaurant_name    0
order_date         0
order_value        0
discount_used      0
payment_method     0
dtype: int64
0
payment_method
Cash    8384
Card    8364
UPI     8252
Name: count, dtype: int64
discount_used
No     12509
Yes    12491
Name: count, dtype: int64


In [ ]:
# Clean orders
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])
#df_orders = df_orders.drop('order_id', axis=1)
df_orders['day_of_week'] = df_orders['order_date'].dt.day_name()

# Verify cleaning
print(df_orders.dtypes)
print(df_orders.shape)
print(df_orders.head())

restaurant_name            object
order_date         datetime64[ns]
order_value               float64
discount_used              object
payment_method             object
day_of_week                object
dtype: object
(25000, 6)
           restaurant_name order_date  order_value discount_used  \
0           TBC Sky Lounge 2025-05-06       201.87            No   
1  KC Das - Sweet Paradise 2026-01-08      1392.27            No   
2        Hotel Kadamba Veg 2026-01-22      1358.35           Yes   
3            Cafe Srinidhi 2025-11-10       782.37            No   
4                   Pebble 2025-12-22      1551.09           Yes   

  payment_method day_of_week  
0           Card     Tuesday  
1           Cash    Thursday  
2            UPI    Thursday  
3            UPI      Monday  
4            UPI      Monday  


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/content/drive/MyDrive/uberproject1/ubereats.db')
cursor = conn.cursor()

# Check if restaurants exists
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Existing tables:", tables)

# If restaurants missing - reload it!
if ('restaurants',) not in tables:
    print("Restaurants missing! Reloading...")
    df_restaurants = pd.read_csv(
        '/content/drive/MyDrive/uberproject1/Uber_Eats_data.csv'
    )
    # Quick clean
    df_restaurants['rate'] = df_restaurants['rate'].str.replace(
        '/5', '', regex=False)
    df_restaurants['rate'] = pd.to_numeric(
        df_restaurants['rate'], errors='coerce')
    df_restaurants = df_restaurants.dropna(subset=['rate'])
    df_restaurants['approx_cost(for two people)'] = df_restaurants[
        'approx_cost(for two people)'].str.replace(',', '', regex=False)
    df_restaurants = df_restaurants.drop(
        columns=['phone', 'listed_in(city)'])
    df_restaurants = df_restaurants.drop_duplicates(keep='first')
    df_restaurants = df_restaurants.rename(columns={
        'approx_cost(for two people)': 'approx_cost_fortwo',
        'listed_in(type)': 'restaurant_type'
    })
    df_restaurants.to_sql(
        'restaurants', conn, if_exists='replace', index=False)
    print("Restaurants reloaded!")

# Push orders
df_orders.to_sql('orders', conn, if_exists='replace', index=False)

# Final verify
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Final tables:", cursor.fetchall())

OperationalError: unable to open database file

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
#discount impact on order value.
q1="""SELECT
    discount_used,
    COUNT(*) as total_orders,
    ROUND(SUM(order_value), 2) as total_revenue,
    ROUND(AVG(order_value), 2) as avg_order_value
FROM orders
GROUP BY discount_used
ORDER BY avg_order_value DESC;"""
df_q1 = pd.read_sql_query(q1, conn)
print("Q1 - discount impact:")
print(df_q1)

Q1 - discount impact:
  discount_used  total_orders  total_revenue  avg_order_value
0           Yes         12491    14363510.85          1149.91
1            No         12509    10288535.97           822.49


Business Insight — complete this sentence: "Customers who use discounts spend ₹327 MORE on average, suggesting discounts are key factor in increasing the revenue which maybe based on psychological triggers in customers

In [ ]:
q2="""SELECT
    day_of_week,
    COUNT(*) as total_orders,
    ROUND(SUM(order_value), 2) as total_revenue,
    ROUND(AVG(order_value), 2) as avg_order_value
FROM orders
GROUP BY day_of_week
ORDER BY total_revenue DESC;"""
df_q2 = pd.read_sql_query(q2, conn)
print("Q2 -REVENUE BY DAY :")
print(df_q2)


Q2 -REVENUE BY DAY :
  day_of_week  total_orders  total_revenue  avg_order_value
0      Monday          3613     3601117.87           996.71
1     Tuesday          3596     3547281.47           986.45
2      Friday          3567     3539518.75           992.30
3   Wednesday          3598     3517348.48           977.58
4    Thursday          3623     3509006.41           968.54
5    Saturday          3505     3494384.74           996.97
6      Sunday          3498     3443389.10           984.39


In [ ]:
q3="""SELECT
    restaurant_name,
    COUNT(*) as total_orders,
    ROUND(SUM(order_value), 2) as total_revenue
FROM orders
GROUP BY restaurant_name
ORDER BY total_revenue DESC
LIMIT 10;"""
df_q3 = pd.read_sql_query(q3, conn)
print("Q3 - TOP 10 RESTAURANTS BY REVENUE:")
print(df_q3)

Q3 - TOP 10 RESTAURANTS BY REVENUE:
       restaurant_name  total_orders  total_revenue
0            Bob's Bar            15       19518.14
1            Cake Cafe            19       19335.17
2         Biryani Mane            15       19249.45
3           Hungry Lee            18       19167.53
4        Andhra Grills            19       19153.72
5          Mighty Paws            17       18410.50
6    Delhi Ke Bawarchi            15       18131.68
7              Pingara            15       17954.06
8         Bawarchi Inn            15       17717.43
9  Soup'ermanz Kitchen            13       17350.72


Top restaurant only has 15 orders,
That means restaurants are very evenly distributed!

In [ ]:
# Add month to dataframe
df_orders['month'] = df_orders['order_date'].dt.month_name()

# Update SQLite table with new column!
df_orders.to_sql('orders', conn, if_exists='replace', index=False)
print("Orders table updated with month column!")

Orders table updated with month column!


In [ ]:
q4="""SELECT
    month,
    COUNT(*) as total_orders,
    ROUND(SUM(order_value), 2) as total_revenue,
    ROUND(AVG(order_value), 2) as avg_order_value
FROM orders
GROUP BY month
ORDER BY total_revenue DESC;"""
df_q4 = pd.read_sql_query(q4, conn)
print("Q4 -REVENUE BY MONTH :")
print(df_q4)

Q4 -REVENUE BY MONTH :
        month  total_orders  total_revenue  avg_order_value
0     January          2642     2630759.98           995.75
1    December          2664     2580580.57           968.69
2         May          2592     2573888.68           993.01
3        July          2613     2567747.54           982.68
4     October          2531     2527196.71           998.50
5    November          2478     2451802.43           989.43
6   September          2483     2449075.72           986.34
7      August          2520     2444881.74           970.19
8        June          2422     2374143.00           980.24
9    February          1903     1900989.85           998.94
10      April           152      150980.60           993.29


In [ ]:
# Check date range!as april only has 152 orders.
print(df_orders['order_date'].min())
print(df_orders['order_date'].max())

2025-04-29 00:00:00
2026-02-23 00:00:00


In [ ]:
#April has 152 orders because dataset starts April 29 — only 2 days!, not low demand.

In [ ]:
q5="""SELECT
    payment_method,
    discount_used,
    COUNT(*) as total_orders,
    ROUND(SUM(order_value), 2) as total_revenue
FROM orders
GROUP BY payment_method, discount_used
ORDER BY total_revenue DESC;"""
df_q5 = pd.read_sql_query(q5, conn)
print("Q5 -REVENUE BY PAYMENT METHOD AND DISCOUNT:")
print(df_q5)

Q5 -REVENUE BY PAYMENT METHOD AND DISCOUNT:
  payment_method discount_used  total_orders  total_revenue
0           Card           Yes          4204     4857263.33
1           Cash           Yes          4203     4815429.35
2            UPI           Yes          4084     4690818.17
3            UPI            No          4168     3443603.18
4           Cash            No          4181     3429996.80
5           Card            No          4160     3414935.99
